In [1]:
# ============================================
# CELL 1: Install Required Packages
# ============================================

print("=" * 80)
print("INSTALLING REQUIRED PACKAGES")
print("=" * 80)

# Install packages with exact versions for compatibility
!pip install -q -U bitsandbytes==0.49.1
!pip install -q -U peft==0.18.1
!pip install -q -U trl==0.27.0
!pip install -q -U accelerate==1.12.0
!pip install -q -U datasets==4.5.0
!pip install -q -U transformers==4.57.5

print("\n✅ All packages installed successfully!")
print("=" * 80)

INSTALLING REQUIRED PACKAGES
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 32.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.5/532.5 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 31.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 122.2 MB/s eta 0:00:0000:010:01

✅ All packages installed successfully!


In [2]:
# ============================================
# CELL 2: Import Required Libraries
# ============================================

print("=" * 80)
print("IMPORTING LIBRARIES")
print("=" * 80)

import os
import torch
import transformers
from datasets import load_dataset, concatenate_datasets, Dataset
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig, AutoPeftModelForCausalLM
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import BitsAndBytesConfig, GemmaTokenizer
import random

print("\n📦 Library versions:")
print(f"   - Transformers: {transformers.__version__}")
print(f"   - PyTorch: {torch.__version__}")
print(f"   - CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   - GPU: {torch.cuda.get_device_name(0)}")

print("\n✅ All libraries imported successfully!")
print("=" * 80)

IMPORTING LIBRARIES


2026-01-27 10:49:57.835002: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769510998.003101      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769510998.050512      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769510998.450912      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769510998.450976      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769510998.450979      55 computation_placer.cc:177] computation placer alr


📦 Library versions:
   - Transformers: 4.57.5
   - PyTorch: 2.8.0+cu126
   - CUDA available: True
   - GPU: Tesla P100-PCIE-16GB

✅ All libraries imported successfully!


In [5]:
# ============================================
# CELL 3: Set HuggingFace Token
# ============================================

print("=" * 80)
print("SETTING UP HUGGINGFACE AUTHENTICATION")
print("=" * 80)

from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")

print("\n✅ HuggingFace token configured successfully!")
print("=" * 80)

SETTING UP HUGGINGFACE AUTHENTICATION

✅ HuggingFace token configured successfully!


In [6]:
# ============================================
# CELL 4: Load and Format Sales Dataset
# ============================================

print("=" * 80)
print("LOADING SALES/MARKETING DATASET")
print("=" * 80)

# Load the sales conversations dataset
print("\n⏳ Loading dataset from goendalf666/sales-conversations...")
sales_ds = load_dataset("goendalf666/sales-conversations", split="train")

print(f"✅ Dataset loaded: {len(sales_ds)} examples")
print(f"\n📋 Original columns: {sales_ds.column_names}")
print(f"\n🔍 Examining first example structure:")
print("-" * 80)
for key, value in sales_ds[0].items():
    print(f"{key}: {str(value)[:200]}...")
print("-" * 80)

# Show a few complete examples to understand the structure
print(f"\n📝 Sample Examples (first 2):")
print("=" * 80)
for i in range(min(2, len(sales_ds))):
    print(f"\nExample {i+1}:")
    print(sales_ds[i])
    print("-" * 80)

print("\n✅ Dataset examination complete!")
print("=" * 80)

LOADING SALES/MARKETING DATASET

⏳ Loading dataset from goendalf666/sales-conversations...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-facf96cb8c12ba(…):   0%|          | 0.00/2.64M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3412 [00:00<?, ? examples/s]

✅ Dataset loaded: 3412 examples

📋 Original columns: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19']

🔍 Examining first example structure:
--------------------------------------------------------------------------------
0: Customer: Hi, Im interested in purchasing a new smartphone. Can you help me choose the best one?...
1: Salesman: Of course, Id be happy to assist you. May I ask what features are most important to you in a smartphone?...
2: Customer: Im looking for a laptop that can handle graphic design software. Any recommendations?...
3: Salesman: Absolutely! In order to find the best laptop for your needs, could you provide me with more details about the specific graphic design software youll be using?...
4: Customer: Im considering upgrading my companys server. What options do you have available?...
5: Salesman: Thank you for considering us for your server upgrade. To better understand your requirements, could you pl

In [7]:
# ============================================
# CELL 5: Format Sales Dataset to Instruction-Response
# ============================================

print("=" * 80)
print("FORMATTING SALES DATASET")
print("=" * 80)

def extract_conversation_pairs(example):
    """
    Extract Customer-Salesman pairs from the conversation format.
    Creates multiple instruction-response pairs from each conversation.
    """
    pairs = []
    
    # Iterate through all column keys
    for i in range(0, 20, 2):  # Step by 2 (customer at even, salesman at odd)
        customer_key = str(i)
        salesman_key = str(i + 1)
        
        customer_msg = example.get(customer_key)
        salesman_msg = example.get(salesman_key)
        
        # Stop if we hit None values
        if customer_msg is None or salesman_msg is None:
            break
        
        # Extract the actual message (remove "Customer: " and "Salesman: " prefixes)
        if customer_msg.startswith("Customer: "):
            instruction = customer_msg[10:]  # Remove "Customer: "
        else:
            instruction = customer_msg
        
        if salesman_msg.startswith("Salesman: "):
            response = salesman_msg[10:]  # Remove "Salesman: "
        else:
            response = salesman_msg
        
        pairs.append({
            "instruction": instruction.strip(),
            "response": response.strip()
        })
    
    return pairs

# Process all examples
print("\n⏳ Extracting instruction-response pairs...")
all_pairs = []

for example in sales_ds:
    pairs = extract_conversation_pairs(example)
    all_pairs.extend(pairs)

# Convert to HuggingFace Dataset
sales_formatted = Dataset.from_list(all_pairs)

print(f"\n✅ Formatting complete!")
print(f"   - Original conversations: {len(sales_ds)}")
print(f"   - Extracted pairs: {len(sales_formatted)}")
print(f"   - Columns: {sales_formatted.column_names}")

# Verify format
print("\n" + "=" * 80)
print("VERIFICATION - Sample Formatted Examples")
print("=" * 80)

for i in range(min(3, len(sales_formatted))):
    print(f"\n📝 Example {i+1}:")
    print(f"INSTRUCTION: {sales_formatted[i]['instruction'][:150]}...")
    print(f"RESPONSE: {sales_formatted[i]['response'][:150]}...")
    print("-" * 80)

# Check for None or empty values
none_instructions = sum(1 for ex in sales_formatted if ex["instruction"] is None or ex["instruction"] == "")
none_responses = sum(1 for ex in sales_formatted if ex["response"] is None or ex["response"] == "")

print(f"\n🔍 Data Quality Check:")
print(f"   - None/Empty instructions: {none_instructions}")
print(f"   - None/Empty responses: {none_responses}")

if none_instructions == 0 and none_responses == 0:
    print("   ✅ All data is clean and ready!")
else:
    print("   ⚠️ WARNING: Some None/Empty values detected!")

print("\n" + "=" * 80)
print("✅ SALES DATASET READY FOR TRAINING!")
print("=" * 80)

FORMATTING SALES DATASET

⏳ Extracting instruction-response pairs...

✅ Formatting complete!
   - Original conversations: 3412
   - Extracted pairs: 20927
   - Columns: ['instruction', 'response']

VERIFICATION - Sample Formatted Examples

📝 Example 1:
INSTRUCTION: Hi, Im interested in purchasing a new smartphone. Can you help me choose the best one?...
RESPONSE: Of course, Id be happy to assist you. May I ask what features are most important to you in a smartphone?...
--------------------------------------------------------------------------------

📝 Example 2:
INSTRUCTION: Im looking for a laptop that can handle graphic design software. Any recommendations?...
RESPONSE: Absolutely! In order to find the best laptop for your needs, could you provide me with more details about the specific graphic design software youll b...
--------------------------------------------------------------------------------

📝 Example 3:
INSTRUCTION: Im considering upgrading my companys server. What options

In [9]:
# ============================================
# CELL 6: Load Previous Checkpoint
# ============================================

print("=" * 80)
print("LOADING PREVIOUS CHECKPOINT")
print("=" * 80)

# Your exact checkpoint name
checkpoint_name = "gemma-checkpoint-6508-20260123-1301"

# Look for the checkpoint in Kaggle input directory
import glob

print(f"\n🔍 Searching for checkpoint: {checkpoint_name}")

# Search in /kaggle/input/
checkpoint_paths = glob.glob(f"/kaggle/input/*{checkpoint_name}*")

if checkpoint_paths:
    checkpoint_dir = checkpoint_paths[0]
    print(f"✅ Found checkpoint at: {checkpoint_dir}")
else:
    print(f"❌ Checkpoint not found in /kaggle/input/")
    print(f"\n💡 Please make sure you:")
    print(f"   1. Added the checkpoint as a dataset to this notebook")
    print(f"   2. The dataset name contains: {checkpoint_name}")
    checkpoint_dir = None

# If found, verify checkpoint contents
if checkpoint_dir:
    print(f"\n📂 Checkpoint contents:")
    checkpoint_files = os.listdir(checkpoint_dir)
    for f in checkpoint_files:
        file_path = os.path.join(checkpoint_dir, f)
        if os.path.isfile(file_path):
            size = os.path.getsize(file_path) / (1024**2)
            print(f"   • {f} ({size:.1f} MB)")
    
    # Check for critical files
    critical_files = ['adapter_model.safetensors', 'adapter_config.json']
    all_present = all(f in checkpoint_files for f in critical_files)
    
    if all_present:
        print(f"\n✅ All critical checkpoint files present!")
    else:
        print(f"\n⚠️ WARNING: Some checkpoint files may be missing!")

print("\n" + "=" * 80)

LOADING PREVIOUS CHECKPOINT

🔍 Searching for checkpoint: gemma-checkpoint-6508-20260123-1301
✅ Found checkpoint at: /kaggle/input/gemma-checkpoint-6508-20260123-1301

📂 Checkpoint contents:
   • adapter_model.safetensors (37.4 MB)
   • trainer_state.json (0.1 MB)
   • training_args.bin (0.0 MB)
   • adapter_config.json (0.0 MB)
   • README.md (0.0 MB)
   • tokenizer.json (32.8 MB)
   • tokenizer_config.json (0.0 MB)
   • scheduler.pt (0.0 MB)
   • special_tokens_map.json (0.0 MB)
   • optimizer.pt (20.2 MB)
   • rng_state.pth (0.0 MB)
   • tokenizer.model (4.0 MB)

✅ All critical checkpoint files present!



In [10]:
# ============================================
# CELL 7: Load Base Model + Checkpoint Adapter
# ============================================

print("=" * 80)
print("LOADING MODEL WITH YOUR CHECKPOINT")
print("=" * 80)

# Configure quantization for efficient training
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# Base model ID
model_id = "google/gemma-2b"
checkpoint_dir = "/kaggle/input/gemma-checkpoint-6508-20260123-1301"

print(f"\n⏳ Loading base model: {model_id}")
print(f"   This may take 1-2 minutes...")

# Load tokenizer from checkpoint (it has your trained tokenizer)
tokenizer = AutoTokenizer.from_pretrained(
    checkpoint_dir,
    token=os.environ["HF_TOKEN"]
)

# Load base model
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map={"": 0},
    token=os.environ["HF_TOKEN"]
)

print(f"✅ Base model loaded!")

# Load your checkpoint adapter on top of base model
print(f"\n⏳ Loading your checkpoint adapter...")
from peft import PeftModel

model = PeftModel.from_pretrained(
    model,
    checkpoint_dir,
    is_trainable=True  # Important: make it trainable for continued training
)

print(f"✅ Checkpoint adapter loaded successfully!")
print(f"\n📊 Model Configuration:")
print(f"   - Base Model: {model_id}")
print(f"   - Checkpoint: gemma-checkpoint-6508-20260123-1301")
print(f"   - Device: {model.device}")
print(f"   - Trainable: Yes")

print("\n" + "=" * 80)
print("✅ MODEL READY FOR CONTINUED TRAINING!")
print("=" * 80)

LOADING MODEL WITH YOUR CHECKPOINT

⏳ Loading base model: google/gemma-2b
   This may take 1-2 minutes...


config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/13.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/67.1M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

✅ Base model loaded!

⏳ Loading your checkpoint adapter...
✅ Checkpoint adapter loaded successfully!

📊 Model Configuration:
   - Base Model: google/gemma-2b
   - Checkpoint: gemma-checkpoint-6508-20260123-1301
   - Device: cuda:0
   - Trainable: Yes

✅ MODEL READY FOR CONTINUED TRAINING!


In [11]:
# ============================================
# CELL 8: Configure LoRA for Continued Training
# ============================================

print("=" * 80)
print("CONFIGURING LORA FOR CONTINUED TRAINING")
print("=" * 80)

# The model already has LoRA adapters from your checkpoint
# We just need to ensure they're trainable and verify the configuration

print(f"\n📋 Current LoRA Configuration:")
print(f"   - LoRA Rank (r): {model.peft_config['default'].r}")
print(f"   - Target Modules: {model.peft_config['default'].target_modules}")
print(f"   - LoRA Alpha: {model.peft_config['default'].lora_alpha}")
print(f"   - LoRA Dropout: {model.peft_config['default'].lora_dropout}")

# Verify model is trainable
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"\n📊 Model Parameters:")
print(f"   - Total parameters: {total_params:,}")
print(f"   - Trainable parameters: {trainable_params:,}")
print(f"   - Trainable %: {100 * trainable_params / total_params:.2f}%")

# Verify adapter is active and trainable
print(f"\n✅ Adapter Status:")
print(f"   - Active adapter: {model.active_adapter}")
print(f"   - Trainable: {model.peft_config['default'].inference_mode == False}")

print("\n" + "=" * 80)
print("✅ LORA CONFIGURATION VERIFIED!")
print("=" * 80)

CONFIGURING LORA FOR CONTINUED TRAINING

📋 Current LoRA Configuration:
   - LoRA Rank (r): 8
   - Target Modules: {'up_proj', 'o_proj', 'gate_proj', 'q_proj', 'down_proj', 'k_proj', 'v_proj'}
   - LoRA Alpha: 8
   - LoRA Dropout: 0.0

📊 Model Parameters:
   - Total parameters: 1,525,073,920
   - Trainable parameters: 9,805,824
   - Trainable %: 0.64%

✅ Adapter Status:
   - Active adapter: default
   - Trainable: True

✅ LORA CONFIGURATION VERIFIED!


In [12]:
# ============================================
# CELL 9: Format Sales Dataset for Training
# ============================================

print("=" * 80)
print("FORMATTING SALES DATASET FOR TRAINING")
print("=" * 80)

def format_instruction(sample):
    """Format samples into instruction-response format for training"""
    return f"""### Instruction:
{sample['instruction']}

### Response:
{sample['response']}"""

print("\n⏳ Applying training format to sales dataset...")
print(f"   Dataset size before formatting: {len(sales_formatted)}")

# Apply formatting to all samples
sales_train = sales_formatted.map(
    lambda x: {"text": format_instruction(x)},
    remove_columns=sales_formatted.column_names
)

print(f"✅ Dataset formatted!")
print(f"   Dataset size after formatting: {len(sales_train)}")
print(f"   Column: {sales_train.column_names}")

# Show sample formatted example
print("\n" + "=" * 80)
print("SAMPLE FORMATTED TRAINING EXAMPLE")
print("=" * 80)
print(sales_train[0]['text'][:400] + "...")
print("=" * 80)

# Shuffle the dataset for better training
sales_train = sales_train.shuffle(seed=42)
print(f"\n✅ Dataset shuffled with seed=42")

print("\n" + "=" * 80)
print("✅ SALES DATASET READY FOR TRAINING!")
print(f"   Total training samples: {len(sales_train):,}")
print("=" * 80)

FORMATTING SALES DATASET FOR TRAINING

⏳ Applying training format to sales dataset...
   Dataset size before formatting: 20927


Map:   0%|          | 0/20927 [00:00<?, ? examples/s]

✅ Dataset formatted!
   Dataset size after formatting: 20927
   Column: ['text']

SAMPLE FORMATTED TRAINING EXAMPLE
### Instruction:
Hi, Im interested in purchasing a new smartphone. Can you help me choose the best one?

### Response:
Of course, Id be happy to assist you. May I ask what features are most important to you in a smartphone?...

✅ Dataset shuffled with seed=42

✅ SALES DATASET READY FOR TRAINING!
   Total training samples: 20,927


In [16]:
# ============================================
# CELL 10: Configure Training Arguments
# ============================================

print("=" * 80)
print("CONFIGURING TRAINING ARGUMENTS")
print("=" * 80)

from transformers import TrainingArguments

# Output directory for this training session
output_dir = "/kaggle/working/gemma-2b-sales-continued"

# IMPORTANT: Lower learning rate to prevent catastrophic forgetting
# Original training used 2e-4, we'll use 5e-5 (2.5x lower)
training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,  # Effective batch size = 8
    num_train_epochs=1,  # Only 1 epoch to prevent forgetting
    learning_rate=5e-5,  # REDUCED from 2e-4 to preserve previous knowledge
    fp16=False,
    bf16=True,
    logging_steps=50,
    save_strategy="steps",
    save_steps=100,  # Save every 500 steps
    save_total_limit=3,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_steps=100,
    max_grad_norm=0.3,
    group_by_length=False,
    report_to="none",
    push_to_hub=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    remove_unused_columns=True,
)

print(f"\n⚙️ Training Configuration:")
print(f"   📂 Output: {output_dir}")
print(f"   🎓 Learning Rate: {training_args.learning_rate} (2.5x LOWER to prevent forgetting)")
print(f"   📊 Effective Batch Size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"   🔄 Epochs: {training_args.num_train_epochs}")
print(f"   💾 Save Every: {training_args.save_steps} steps")
print(f"   📈 Training Samples: {len(sales_train):,}")

# Calculate estimated training steps
steps_per_epoch = len(sales_train) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)
total_steps = steps_per_epoch * training_args.num_train_epochs

print(f"\n📊 Training Statistics:")
print(f"   - Steps per epoch: {steps_per_epoch}")
print(f"   - Total training steps: {total_steps}")
print(f"   - Expected checkpoints: ~{total_steps // training_args.save_steps}")
print(f"   - Estimated time: ~45-60 minutes on P100 GPU")

print("\n" + "=" * 80)
print("✅ TRAINING ARGUMENTS CONFIGURED!")
print("=" * 80)
print("\n🔑 KEY STRATEGY FOR PREVENTING FORGETTING:")
print("   ✓ Lower learning rate (5e-5 vs 2e-4 original)")
print("   ✓ Only 1 epoch on new data")
print("   ✓ Cosine LR schedule with warmup")
print("   ✓ Continued from your trained checkpoint")
print("=" * 80)

CONFIGURING TRAINING ARGUMENTS

⚙️ Training Configuration:
   📂 Output: /kaggle/working/gemma-2b-sales-continued
   🎓 Learning Rate: 5e-05 (2.5x LOWER to prevent forgetting)
   📊 Effective Batch Size: 8
   🔄 Epochs: 1
   💾 Save Every: 100 steps
   📈 Training Samples: 20,927

📊 Training Statistics:
   - Steps per epoch: 2615
   - Total training steps: 2615
   - Expected checkpoints: ~26
   - Estimated time: ~45-60 minutes on P100 GPU

✅ TRAINING ARGUMENTS CONFIGURED!

🔑 KEY STRATEGY FOR PREVENTING FORGETTING:
   ✓ Lower learning rate (5e-5 vs 2e-4 original)
   ✓ Only 1 epoch on new data
   ✓ Cosine LR schedule with warmup
   ✓ Continued from your trained checkpoint


In [17]:
# ============================================
# CELL 11: Initialize SFTTrainer
# ============================================

print("=" * 80)
print("INITIALIZING SFTTRAINER")
print("=" * 80)

from trl import SFTTrainer, SFTConfig

# Create SFTConfig (without deprecated parameters)
sft_config = SFTConfig(
    output_dir=output_dir,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=5e-5,  # Lower LR to prevent forgetting
    fp16=False,
    bf16=True,
    logging_steps=50,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_steps=100,
    max_grad_norm=0.3,
    group_by_length=False,
    report_to="none",
    push_to_hub=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
)

print(f"\n⏳ Initializing SFTTrainer...")

trainer = SFTTrainer(
    model=model,
    train_dataset=sales_train,
    args=sft_config,
)

print(f"✅ Trainer initialized successfully!")

print(f"\n📊 Trainer Configuration:")
print(f"   - Model: Gemma 2B + Your Checkpoint")
print(f"   - Training dataset: {len(sales_train):,} sales examples")
print(f"   - Device: {model.device}")
print(f"   - Learning Rate: 5e-5 (optimized for continued training)")

print("\n" + "=" * 80)
print("✅ READY TO START TRAINING!")
print("=" * 80)

INITIALIZING SFTTRAINER

⏳ Initializing SFTTrainer...


Adding EOS to train dataset:   0%|          | 0/20927 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/20927 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/20927 [00:00<?, ? examples/s]

✅ Trainer initialized successfully!

📊 Trainer Configuration:
   - Model: Gemma 2B + Your Checkpoint
   - Training dataset: 20,927 sales examples
   - Device: cuda:0
   - Learning Rate: 5e-5 (optimized for continued training)

✅ READY TO START TRAINING!


In [18]:
# ============================================
# CELL 12: Start Training
# ============================================

print("=" * 80)
print("STARTING CONTINUED TRAINING ON SALES DATA")
print("=" * 80)

print(f"\n📊 Training Overview:")
print(f"   - Starting from: checkpoint-6508")
print(f"   - New dataset: Sales/Marketing (20,927 examples)")
print(f"   - Total steps: 2,615")
print(f"   - Checkpoints every: 100 steps")
print(f"   - Learning rate: 5e-5 (reduced to preserve previous knowledge)")
print(f"   - Expected duration: ~45-60 minutes")

print(f"\n⏳ Training started...")
print(f"   You can monitor progress below.")
print(f"   Checkpoints will be saved to: {output_dir}")
print("\n" + "=" * 80)

# Start training
trainer.train()

print("\n" + "=" * 80)
print("✅ TRAINING COMPLETED SUCCESSFULLY!")
print("=" * 80)

STARTING CONTINUED TRAINING ON SALES DATA

📊 Training Overview:
   - Starting from: checkpoint-6508
   - New dataset: Sales/Marketing (20,927 examples)
   - Total steps: 2,615
   - Checkpoints every: 100 steps
   - Learning rate: 5e-5 (reduced to preserve previous knowledge)
   - Expected duration: ~45-60 minutes

⏳ Training started...
   You can monitor progress below.
   Checkpoints will be saved to: /kaggle/working/gemma-2b-sales-continued



huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Step,Training Loss
50,1.815300
100,1.529800
150,1.285600
200,1.214600
250,1.170800
300,1.146200
350,1.117900
400,1.136800
450,1.124800
500,1.087600



✅ TRAINING COMPLETED SUCCESSFULLY!


In [19]:
# ============================================
# CELL 13: Save the Final Model
# ============================================

print("=" * 80)
print("SAVING TRAINED MODEL")
print("=" * 80)

# Define the final model save path
final_model_path = "/kaggle/working/gemma-2b-sales-final"

print(f"\n💾 Saving model to: {final_model_path}")
print("\nThis includes:")
print("   - LoRA adapter weights")
print("   - Training configuration")
print("   - Tokenizer files")
print("   - Model configuration")

# Save the model and tokenizer
trainer.model.save_pretrained(final_model_path)
tokenizer.save_pretrained(final_model_path)

print("\n✅ Model saved successfully!")
print("=" * 80)


SAVING TRAINED MODEL

💾 Saving model to: /kaggle/working/gemma-2b-sales-final

This includes:
   - LoRA adapter weights
   - Training configuration
   - Tokenizer files
   - Model configuration

✅ Model saved successfully!


In [20]:
# ============================================
# CELL 14: Create Zip File for Download
# ============================================

print("=" * 80)
print("CREATING ZIP FILE FOR DOWNLOAD")
print("=" * 80)

import shutil
import os

# Define zip file name and path
zip_filename = "gemma-2b-sales-trained"
zip_path = f"/kaggle/working/{zip_filename}"

print(f"\n📦 Creating zip file: {zip_filename}.zip")
print(f"   Source: {final_model_path}")
print(f"   This may take a few minutes...")

# Create the zip file
shutil.make_archive(zip_path, 'zip', final_model_path)

# Get file size
zip_file_path = f"{zip_path}.zip"
file_size_mb = os.path.getsize(zip_file_path) / (1024 * 1024)

print(f"\n✅ Zip file created successfully!")
print(f"\n📊 File Details:")
print(f"   - File name: {zip_filename}.zip")
print(f"   - Location: {zip_file_path}")
print(f"   - Size: {file_size_mb:.2f} MB")

print("\n" + "=" * 80)
print("📥 READY FOR DOWNLOAD")
print("=" * 80)
print("\n💡 To download:")
print("   1. Click on 'Output' section on the right panel")
print("   2. Find 'gemma-2b-sales-trained.zip'")
print("   3. Click the download icon")
print("\nAlternatively, the model files are available in:")
print(f"   {final_model_path}")
print("=" * 80)


CREATING ZIP FILE FOR DOWNLOAD

📦 Creating zip file: gemma-2b-sales-trained.zip
   Source: /kaggle/working/gemma-2b-sales-final
   This may take a few minutes...

✅ Zip file created successfully!

📊 File Details:
   - File name: gemma-2b-sales-trained.zip
   - Location: /kaggle/working/gemma-2b-sales-trained.zip
   - Size: 21.95 MB

📥 READY FOR DOWNLOAD

💡 To download:
   1. Click on 'Output' section on the right panel
   2. Find 'gemma-2b-sales-trained.zip'
   3. Click the download icon

Alternatively, the model files are available in:
   /kaggle/working/gemma-2b-sales-final


In [21]:
# ============================================
# CELL 15: Create Zip of Last Checkpoint
# ============================================

print("=" * 80)
print("CREATING ZIP FILE FOR LAST CHECKPOINT")
print("=" * 80)

import shutil
import os
import glob

# Find the last checkpoint
checkpoint_dir = "/kaggle/working/gemma-2b-sales-continued"
checkpoints = glob.glob(os.path.join(checkpoint_dir, "checkpoint-*"))

if checkpoints:
    # Sort checkpoints by number to get the last one
    checkpoints.sort(key=lambda x: int(x.split("-")[-1]))
    last_checkpoint = checkpoints[-1]
    checkpoint_name = os.path.basename(last_checkpoint)
    
    print(f"\n📂 Last checkpoint found: {checkpoint_name}")
    print(f"   Location: {last_checkpoint}")
    
    # Define zip file name and path
    zip_filename = f"gemma-2b-sales-{checkpoint_name}"
    zip_path = f"/kaggle/working/{zip_filename}"
    
    print(f"\n📦 Creating zip file: {zip_filename}.zip")
    print(f"   This may take a few minutes...")
    
    # Create the zip file
    shutil.make_archive(zip_path, 'zip', last_checkpoint)
    
    # Get file size
    zip_file_path = f"{zip_path}.zip"
    file_size_mb = os.path.getsize(zip_file_path) / (1024 * 1024)
    
    print(f"\n✅ Zip file created successfully!")
    print(f"\n📊 File Details:")
    print(f"   - Checkpoint: {checkpoint_name}")
    print(f"   - File name: {zip_filename}.zip")
    print(f"   - Location: {zip_file_path}")
    print(f"   - Size: {file_size_mb:.2f} MB")
    
    print("\n" + "=" * 80)
    print("📥 CHECKPOINT READY FOR DOWNLOAD")
    print("=" * 80)
    print("\n💡 To download:")
    print("   1. Click on 'Output' section on the right panel")
    print(f"   2. Find '{zip_filename}.zip'")
    print("   3. Click the download icon")
    print("\n📌 Note: This checkpoint can be used to resume training")
    print("   or for inference without merging.")
    print("=" * 80)
    
else:
    print("\n⚠️  No checkpoints found in the output directory!")
    print(f"   Searched in: {checkpoint_dir}")
    print("\n💡 Make sure training has completed and saved checkpoints.")
    print("=" * 80)

CREATING ZIP FILE FOR LAST CHECKPOINT

📂 Last checkpoint found: checkpoint-2616
   Location: /kaggle/working/gemma-2b-sales-continued/checkpoint-2616

📦 Creating zip file: gemma-2b-sales-checkpoint-2616.zip
   This may take a few minutes...

✅ Zip file created successfully!

📊 File Details:
   - Checkpoint: checkpoint-2616
   - File name: gemma-2b-sales-checkpoint-2616.zip
   - Location: /kaggle/working/gemma-2b-sales-checkpoint-2616.zip
   - Size: 39.37 MB

📥 CHECKPOINT READY FOR DOWNLOAD

💡 To download:
   1. Click on 'Output' section on the right panel
   2. Find 'gemma-2b-sales-checkpoint-2616.zip'
   3. Click the download icon

📌 Note: This checkpoint can be used to resume training
   or for inference without merging.


In [22]:
# ============================================
# CELL 16: Setup Model for Inference
# ============================================

print("=" * 80)
print("SETTING UP MODEL FOR INFERENCE")
print("=" * 80)

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import AutoPeftModelForCausalLM

# Define the model path (use the final saved model)
model_path = "/kaggle/working/gemma-2b-sales-final"

print(f"\n📥 Loading model from: {model_path}")
print("   This may take a moment...")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Load the model with PEFT adapters
model = AutoPeftModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Set model to evaluation mode
model.eval()

print("\n✅ Model loaded successfully!")
print(f"   Device: {model.device}")
print(f"   Model type: Gemma 2B with LoRA adapters")
print("=" * 80)


SETTING UP MODEL FOR INFERENCE

📥 Loading model from: /kaggle/working/gemma-2b-sales-final
   This may take a moment...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]


✅ Model loaded successfully!
   Device: cuda:0
   Model type: Gemma 2B with LoRA adapters


In [ ]:
# ============================================
# CELL 17: Interactive Chat Session
# ============================================

print("=" * 80)
print("INTERACTIVE CHAT SESSION - MARKETING AGENT")
print("=" * 80)

import torch

def generate_response(instruction, max_length=256, temperature=0.7, top_p=0.9):
    """
    Generate a response from the model given an instruction
    """
    # Format the prompt
    prompt = f"""### Instruction:
{instruction}

### Response:"""
    
    # Tokenize
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_length,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,
        )
    
    # Decode and extract only the response part
    full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract only the response after "### Response:"
    if "### Response:" in full_response:
        response = full_response.split("### Response:")[-1].strip()
    else:
        response = full_response
    
    return response

def chat_session():
    """
    Interactive chat session
    """
    print("\n🤖 Marketing Agent Chat Session Started!")
    print("=" * 80)
    print("\n💡 Commands:")
    print("   - Type 'exit' or 'quit' to end session")
    print("   - Type 'clear' to clear conversation history")
    print("   - Type 'examples' to see test scenarios")
    print("\n" + "=" * 80)
    
    conversation_history = []
    
    while True:
        # Get user input
        print("\n" + "─" * 80)
        user_input = input("\n👤 You: ").strip()
        
        # Check for commands
        if user_input.lower() in ['exit', 'quit']:
            print("\n👋 Ending chat session. Goodbye!")
            print("=" * 80)
            break
        
        if user_input.lower() == 'clear':
            conversation_history = []
            print("\n🗑️  Conversation history cleared!")
            continue
        
        if user_input.lower() == 'examples':
            print("\n📋 Test Scenarios:")
            print("\n1. PRODUCT RECOMMENDATION:")
            print("   'Im looking for a CRM solution for my small business'")
            print("\n2. OBJECTION HANDLING:")
            print("   'Your product is too expensive compared to competitors'")
            print("\n3. FEATURE INQUIRY:")
            print("   'What features does your software include?'")
            print("\n4. CLOSING:")
            print("   'Im interested but need to think about it'")
            print("\n5. TECHNICAL QUESTION:")
            print("   'How does your product integrate with other tools?'")
            continue
        
        if not user_input:
            print("⚠️  Please enter a message or command.")
            continue
        
        # Generate response
        print("\n🤖 Agent: ", end="", flush=True)
        
        try:
            response = generate_response(user_input)
            print(response)
            
            # Add to history
            conversation_history.append({
                "user": user_input,
                "agent": response
            })
            
        except Exception as e:
            print(f"\n❌ Error generating response: {str(e)}")
            print("Please try again with a different message.")

# Start the chat session
chat_session()

INTERACTIVE CHAT SESSION - MARKETING AGENT

🤖 Marketing Agent Chat Session Started!

💡 Commands:
   - Type 'exit' or 'quit' to end session
   - Type 'clear' to clear conversation history
   - Type 'examples' to see test scenarios


────────────────────────────────────────────────────────────────────────────────



👤 You:  Im looking for a marketing automation tool for my startup



🤖 Agent: Thats great! Can you tell me more about your startup and what specific features you need in a marketing automation tool?

────────────────────────────────────────────────────────────────────────────────



👤 You:  Can you recommend a solution for managing customer relationships?



🤖 Agent: Certainly! Building strong and lasting relationships with customers is crucial. There are several strategies you can consider, such as providing personalized attention, offering exclusive benefits, and actively engaging with them through various touchpoints. By focusing on building meaningful connections and exceeding their expectations, you can establish a solid foundation for long-term success.

────────────────────────────────────────────────────────────────────────────────


...


🤖 Agent: Hello, [Name]! I noticed you are looking for a cloud storage solution for your small to medium business. CloudSync Solutions offers a comprehensive platform that solves many of your pain points. With our robust features like real-time collaboration, 5TB storage per user, mobile and desktop apps, and 24/7 customer support, we can help streamline your operations and boost productivity. Plus, we offer a free trial for 30 days, allowing you to try out our services risk-free. Would you like to schedule a demo to discuss how CloudSync Solutions can benefit your business?

────────────────────────────────────────────────────────────────────────────────



👤 You:  Your CloudSync solution company seems good whats the uptime of your business and how cheap is it compared to other solutions?



🤖 Agent: Thank you for your interest in our CloudSync solution. Let me explain how our service compares with others in terms of reliability and pricing. Our solution has a 99.9% uptime guarantee, ensuring that your data is always available and accessible. Additionally, our pricing is extremely competitive, with discounts available for longer term contracts. Let me know if you have any further questions.

────────────────────────────────────────────────────────────────────────────────



👤 You:  and does your cloudsync solution provide migration facility from other platforms?



🤖 Agent: Certainly! Our cloud sync solution offers seamless migration from other platforms. It allows you to seamlessly transfer your data, settings, and files without any hassle. Additionally, our solution also provides integration with popular collaboration tools, ensuring a smooth transition for your team.

────────────────────────────────────────────────────────────────────────────────
